# 0902 3일차

## 0. 파이썬 문법 - 슬라이싱과 튜플 언패킹

### 0-1. 슬라이싱

리스트·배열에서 `[시작:끝]`으로 일부 구간을 잘라내는 문법

```python
x = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])

x[0:7]    # x[:7]  시작 0은 생략 가능  -> 1 ~ 7
x[7:10]   # x[7:]  끝도 생략 가능      -> 8, 9, 10
```

**규칙**
1. 끝 인덱스는 포함하지 않음 (`x[0:7]`은 인덱스 0~6)
2. 시작과 끝은 생략할 수 있음
3. 그래서 `x[:7]`과 `x[7:]`을 이어붙이면 원본이 됨

### 0-2. 튜플 언패킹

여러 값을 괄호 구조대로 나눠서 여러 변수에 한 번에 담는 문법

```python
a, b = 1, 2                       # a=1, b=2
(a, b), (c, d) = (1, 2), (3, 4)   # 괄호 모양대로 나눠 담김
```

**규칙**
- 왼쪽 괄호 구조와 오른쪽 값의 구조가 같아야 함. 안 맞으면 `ValueError`

케라스 데이터셋이 이 문법으로 데이터를 돌려줌

```python
(x_train, y_train), (x_test, y_test) = boston_housing.load_data()
```

## 1. 학습(Training)과 추론(Inference)

**개념**
1. 학습 : 최적의 w, b를 찾아내는 과정 → 순전파 + 역전파 + 갱신
2. 추론 : 이미 찾아둔 w, b를 대입해서 답을 내는 과정 → 순전파만

학습은 w를 만드는 단계, 추론은 그 w를 쓰는 단계임 (1일차 §3-3)

| | 순전파 | 역전파 | 갱신 |
|---|---|---|---|
| **학습** | O | O | O |
| **추론** | O | X | X |

**추론에서 가중치가 바뀌지 않는 이유**
- 가중치 갱신식 $w \leftarrow w - \eta \cdot \frac{\partial L}{\partial w}$ (§cf)이 실행되지 않음
- 갱신 단계 자체가 없어서 구조적으로 학습이 불가능함 (= `model.predict()`가 하는 일)

### 1-1. 학습과 추론의 실행 횟수

1. 학습 : 개발 단계에서 며칠이 걸려도 한 번 끝내면 됨 → 수렴이 중요함
2. 추론 : 서비스가 운영되는 동안 요청마다 매번 실행됨 → 속도가 중요함

실행 횟수가 달라서 중요하게 보는 지표도 다름

### 1-2. NPU

Neural Processing Unit. 신경망 연산에 특화된 프로세서

**추론용 하드웨어가 따로 있는 이유**
- 추론은 정해진 w로 순전파(행렬 곱셈)를 반복하는 것뿐이라 계산 패턴이 단순하고 고정적임
- 역전파가 없어서 그 패턴에만 맞춰 하드웨어를 설계할 수 있음

**역할 분담**
- 보통 학습은 GPU, 추론은 NPU가 담당함
- NPU는 응답 지연(latency), 초당 처리량, 전력당 성능으로 비교함

## 2. train / test 나누는 방법

2일차 §4에서 나누는 이유를 정리했고, 여기서는 나누는 방법을 정리함

**방법의 종류**
1. 값을 직접 적기 : `x_train = np.array([1, ..., 7])` (`keras09_train_test1.py`)
   - 데이터가 적을 때만 가능하고, 오타가 나도 알아채기 어려움
2. 슬라이싱 : `x_train = x[:7]` (`keras09_train_test2.py`)
   - 원본에서 잘라내므로 값이 어긋날 일이 없음
3. 라이브러리 : `train_test_split(x, y, train_size=0.7)` (`keras09_train_test3.py`)
   - 섞어서 자름

1, 2번의 공통 문제는 순서대로 자른다는 것임 (§2-1)

### 2-1. shuffle이 필요한 이유

- x가 1~10인데 앞 7개 / 뒤 3개로 자르면 train은 1~7만 보고, test는 훈련에서 본 적 없는 8, 9, 10이 됨
- 이것은 평가가 아니라 외삽(extrapolation)임
- 결과가 나빠도 과적합 때문인지 범위 밖이라서인지 구분할 수 없음
- 섞어서 자르면 train과 test가 같은 분포를 가짐

### 2-2. train_test_split

```python
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x, y, train_size=0.7, random_state=123)
```

**주요 인자**
1. `train_size` : 훈련 데이터 비중 (기본값 `0.75`)
2. `test_size` : 평가 데이터 비중 (둘 중 하나만 적어도 됨)
3. `shuffle` : 섞을지 여부 (기본값 `True`)
4. `random_state` : 섞는 순서를 고정하는 난수 씨앗 (기본값 없음)

**규칙**
1. 반환 순서는 `x_train, x_test, y_train, y_test`
   - `x_train, y_train, x_test, y_test` 순이 아님. 잘못 받아도 에러가 안 나고 학습만 이상해짐
2. `train_size`와 `test_size`를 같이 쓸 때
   - 둘 다 생략 → `0.75` / `0.25`
   - 합이 1보다 작으면 나머지는 버려짐
   - 합이 1보다 크면 `ValueError`

### 2-3. random_state를 고정하는 이유

- 주지 않으면 실행할 때마다 다른 조합으로 나뉨 → loss가 변해도 모델 덕인지 분할 운인지 구분할 수 없음
- 고정하면 분할을 상수로 묶고 모델만 비교할 수 있음
- 값 자체에는 의미가 없음(`123`이든 `42`든). 같은 값을 계속 쓰는 것이 중요함

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

x = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
y = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])

x_train, x_test, y_train, y_test = train_test_split(x, y, train_size=0.7, random_state=123)
print(x_train)   # [ 6  9  4  2  7 10  3]
print(x_test)    # [5 1 8]

# 순서가 의미 있는 데이터는 shuffle=False -> 앞에서부터 그대로 자름
x_tr3, x_te3, _, _ = train_test_split(x, y, train_size=0.7, shuffle=False)
print(x_tr3)     # [1 2 3 4 5 6 7]

### 2-4. 예외) 시계열 데이터

주가, 매출, 센서 로그처럼 시간 순서 자체가 의미를 갖는 데이터

| | 자르는 방법 | 이유 |
|---|---|---|
| **일반 데이터** | shuffle 후 자름 | train/test가 같은 분포를 갖게 |
| **시계열** | `shuffle=False`, 앞=train / 뒤=test | 미래 정보가 훈련에 섞이지 않게 |

- 섞으면 3월을 보고 2월을 맞히는 모델이 됨 → test 점수는 좋지만 실전에서 재현되지 않음

**시계열이 외삽이 아닌 이유**
- `keras09`는 입력이 1~10이라는 값 자체라서, 8·9·10은 못 본 구간 → 외삽
- 시계열은 입력이 시점 t가 아니라 과거 값들(`y[t-1]`, `y[t-2]`)이라 값의 범위가 비슷함 → 외삽이 아님
- 단, 추세가 있으면 시계열도 외삽이 됨 → 차분(differencing)으로 추세를 제거함

## 3. 시각화

loss는 숫자 하나라서 얼마나 틀렸는지는 알려주지만 어떻게 틀렸는지는 보여주지 않음 → 그래프로 확인함

```python
import matplotlib.pyplot as plt

results = model.predict(x)          # 선을 그리려고 전체 x에 대해 예측

plt.scatter(x, y)                   # 실제 데이터 = 점
plt.plot(x, results, color="red")   # 모델이 그은 선
plt.show()
```

**`predict`에 전체 `x`를 넣는 이유**
- 평가가 아니라 선을 그리는 것이 목적이라 x 전 구간의 예측값이 필요함
- 평가는 `evaluate(x_test, y_test)`로 따로 했으므로 test가 오염되지 않음

**그래프 읽는 법**
1. 점들이 선 주위에 고르게 있음 → 잘 맞춘 것
2. 점들이 선의 한쪽으로 치우침 → 모델이 데이터를 따라가지 못함
3. 유난히 멀리 떨어진 점 → 이상치(outlier)

### 3-1. 노이즈가 있으면 loss가 0이 될 수 없음

- `keras10_scatter1.py` : `y = [1, 2, 3, 4, 7, 5, 7, 8, 6, 10]` → 완벽한 비례가 아님
- `keras10_scatter2.py` : 노이즈를 더 키움 → `loss` 약 4.3
- 어떤 직선을 그어도 모든 점을 지날 수 없어 loss가 0으로 수렴하지 못함
- loss 숫자만 보면 실패처럼 보여도, 그래프로 보면 정상인 경우가 있음

## 4. 실전 데이터셋 불러오기

**불러오는 방식의 종류**
1. sklearn 방식 : x, y를 받은 뒤 직접 분할함
2. 케라스 방식 : train / test가 나뉜 채로 받음

### 4-1. sklearn 방식

```python
from sklearn.datasets import load_diabetes

datasets = load_diabetes()
x = datasets.data       # 특성
y = datasets.target     # 정답

x, y = load_diabetes(return_X_y=True)   # 바로 받고 싶으면
```

- `load_diabetes`는 소문자라 함수이지만, 반환하는 것이 `Bunch`라는 객체라서 `.`으로 접근함

**함수 이름의 접두사**
1. `load_` : sklearn에 포함된 소형 데이터 → 캐시 불필요
2. `fetch_` : 인터넷에서 내려받는 대형 데이터 → 홈 폴더의 `scikit_learn_data`에 캐시
   - 첫 실행만 느리고 인터넷이 필요함. 이후에는 오프라인에서도 동작함
   - `keras11_1_california.py` 맨 위의 SSL 우회 코드도 첫 실행에만 필요함

### 4-2. 케라스 방식

```python
from tensorflow.keras.datasets import boston_housing

(x_train, y_train), (x_test, y_test) = boston_housing.load_data()
```

| 케라스 인자 | 대응하는 sklearn 인자 |
|---|---|
| `test_split=0.2` | `test_size=0.2` |
| `seed=113` | `random_state=113` |

**주의) 반환 순서가 sklearn과 반대**

| | 묶는 기준 | 순서 |
|---|---|---|
| **sklearn** | x끼리 / y끼리 | `x_train, x_test, y_train, y_test` |
| **케라스** | train끼리 / test끼리 | `(x_train, y_train), (x_test, y_test)` |

- 습관대로 받으면 x와 y가 뒤바뀜. 에러가 안 나고 학습만 이상해짐
- 케라스는 `(a, b), (c, d)`처럼 괄호가 두 겹임 (§0-2)

## 5. loss 읽기

### 5-1. fit의 loss와 evaluate의 loss

1. `fit`이 매 epoch 출력하는 loss : `x_train` 기준 (보고 배운 데이터)
2. `evaluate`가 반환하는 loss : `x_test` 기준 (처음 보는 데이터)

| train loss | test loss | 진단 |
|---|---|---|
| 작다 | 작다 | 잘 학습됨 |
| 작다 | **크다** | **과적합** |
| 크다 | 크다 | 과소적합 |

**evaluate가 학습이 아닌 이유**
- 순전파 + 손실 계산만 하고, 역전파와 갱신이 없음
- `batch_size`를 주지 않으면 32씩 나눠 계산함
- 나눠 계산해도 가중치 갱신이 없어서 결과는 같음 (`fit`의 batch와 의미가 다름)

### 5-2. loss를 데이터셋끼리 비교할 수 없는 이유

| 데이터셋 | y 범위 | loss | loss의 제곱근 | 실제 의미 |
|---|---|---|---|---|
| 캘리포니아 | 0.15 ~ 5.0 | 약 0.57 | 약 0.76 | 평균 7만 6천 달러 빗나감 |
| 보스턴 | 5.0 ~ 50.0 | 약 22.9 | 약 4.8 | 평균 4천 8백 달러 빗나감 |
| 당뇨 | 25 ~ 346 | 약 2,384 | 약 48.8 | 진행도 기준 평균 48.8 빗나감 |

- 당뇨의 2,384가 보스턴의 22.9보다 100배 나쁜 것이 아니라, y의 스케일이 다를 뿐임
- `mse`는 단위가 y 단위의 제곱이라 숫자가 커짐
- 제곱근을 씌워 원래 단위로 읽는 것이 RMSE임 (4일차 §2)

### 5-3. 특성 스케일

**특성마다 범위가 다르면 생기는 문제**

캘리포니아 특성별 범위

| 특성 | 범위 |
|---|---|
| `MedInc` (소득) | 0.50 ~ 15.00 |
| `Population` (인구) | 3.00 ~ 35,682.00 |
| `Longitude` (경도) | -124.35 ~ -114.31 |

- `Population`이 수천 배 커서 `y = wx + b`에서 이 항의 영향이 가장 큼

**당뇨 데이터는 이미 정규화되어 있음**

```
age   -0.107 ~ 0.111   평균 -0.000
bmi   -0.090 ~ 0.171   평균 -0.000
```

- 나이가 `-0.107`일 수는 없음 → sklearn이 스케일을 맞춘 값을 제공하는 것
- 당뇨는 그대로 넣어도 되지만, 캘리포니아·보스턴은 정규화(scaling) 없이는 성능이 제한됨
- 이것이 데이터 전처리가 필요한 이유임 (9일차 §4)

## cf) 경사 하강법 갱신식

### MSE에서 제곱을 쓰는 이유

1. 부호 상쇄 방지 : `+3`과 `-3`을 더하면 0이 되어 오차가 없는 것처럼 보임
2. 미분 가능 : 제곱은 매끄러운 곡선이라 어디서든 기울기를 구할 수 있음
   - loss를 미분할 수 있어야 경사 하강법이 동작함
3. 큰 오차에 더 큰 벌점 : `0.5² = 0.25`, `8² = 64` → 크게 틀린 것부터 고침
   - 대신 이상치 하나에 크게 끌려감 → 이상치가 많으면 `mae`를 쓰기도 함

### 가중치 갱신식

$$w \leftarrow w - \eta \cdot \frac{\partial L}{\partial w}$$

**구성 요소**
1. $w$ : 지금 가중치
2. $\eta$ (eta) : 학습률(learning rate) → 한 번에 얼마나 움직일지
3. $\partial L / \partial w$ : 그 지점에서의 기울기
   - 나눗셈이 아니라 "L을 w로 미분한 것"이라는 기호 하나

**기울기를 빼는 이유**
- $\partial L / \partial w$는 loss가 커지는 방향(오르막)을 가리킴 → 반대로 가야 해서 부호가 `-`

| 기울기 | 하는 일 |
|---|---|
| 양수 (오르막) | w를 줄임 |
| 음수 (내리막) | w를 늘림 |
| 0 | 더 움직이지 않음 = 최소 지점 |

**학습률의 크기**
1. 너무 작으면 : 보폭이 작아 학습이 끝나지 않음
2. 너무 크면 : 최소 지점을 건너뛰어 반대편에 착지함 → 발산

### 역전파와 optimizer의 역할 분담

| 학습 1스텝 | 담당 | 이 식에서 |
|---|---|---|
| 3. 역전파 | backpropagation | $\partial L / \partial w$ 를 구함 |
| 4. 갱신 | optimizer | $w - \eta \cdot \partial L / \partial w$ 를 실행 |

- 역전파는 오른쪽 항 하나를 계산할 뿐이고, 식 전체를 실행하는 것은 optimizer임
- `sgd`는 위 식 그대로, `adam`은 관성과 학습률 자동 조절을 더한 개량형